# 04 — Federated Learning

**A³E-DNS Thesis Artifact · Chapter 6 · §6.6, Eqs. 126–130**

This notebook exercises the FL coordinator, the DP-SGD mechanism,
and the geometric-median aggregation. It demonstrates the Byzantine-
robustness property that motivates §6.6.4: outliers do not pull the
aggregate arbitrarily far.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from src.models.fl_coordinator import FLCoordinator, FLCoordinatorConfig

rng = np.random.default_rng(20261130)
dim = 16
n_clients = 20
coord = FLCoordinator(
    FLCoordinatorConfig(rounds=5, clients=n_clients, dp_epsilon=0.68),
    model_dim=dim,
)

for r in range(5):
    updates = [rng.normal(0.0, 0.05, size=dim) for _ in range(n_clients)]
    result = coord.run_round(updates, round_index=r)
    print(f'round {result.round}  used={result.n_updates_used}  '
          f'byz_isolation={result.n_byzantine_isolated}  '
          f'median_norm={result.geometric_median_norm:.4f}')

### Byzantine robustness

Inject one far outlier and observe that the geometric median is not
pulled toward it. The arithmetic mean, by contrast, is.

In [ ]:
from src.models.fl_geometric_median import weighted_geometric_median

rng = np.random.default_rng(20261131)
honest = rng.normal(0.0, 0.1, size=(20, 4))
outlier = np.full((1, 4), 100.0)
points = np.vstack([honest, outlier])

median = weighted_geometric_median(points)
mean = points.mean(axis=0)

print(f'geometric median norm: {np.linalg.norm(median):.4f}')
print(f'arithmetic mean norm : {np.linalg.norm(mean):.4f}')

### DP-SGD noise calibration

The thesis operating point is ε = 0.68 (§6.6.5). The mechanism below calibrates the noise multiplier for that target.

In [ ]:
from src.models.fl_dp_sgd import DPConfig, DPSGDMechanism

mech = DPSGDMechanism(DPConfig(epsilon=0.68, rounds=100))
nm = mech.calibrate()
print(f'calibrated noise multiplier: {nm:.4f}')
print(f'epsilon at that multiplier: {mech.epsilon_at(nm):.4f}')